# IPS 2단계 — 룰 기반(Suricata + ET Open) vs AI 탐지 비교

**목표**: 같은 네트워크 트래픽(pcap)을 룰 기반 IPS 엔진과 AI 모델에 넣고, **같은 흐름(flow) 단위**로 탐지율·오탐률을 비교한다.

| 구분 | 내용 |
|---|---|
| 데이터 | CIC-IDS2017 원본 pcap 하루치 (기본: 금요일 — 봇넷, 포트 스캔, DDoS) + CIC가 배포한 흐름 라벨 |
| 룰 기반 | Suricata (OISF 안정판) + Emerging Threats Open 룰 (`suricata-update`로 실행 당일 최신본) |
| AI 모델 A | Suricata 흐름 기록의 수치 특성 + 그래디언트 부스팅 (CPU) |
| AI 모델 B | 같은 특성 + 다층 신경망 MLP (GPU) |

**핵심 설계**: AI가 쓰는 특성도 **Suricata가 만든 흐름 기록**에서 뽑습니다. 그래서 룰과 AI가 **완전히 같은 흐름 목록**을 놓고 판정합니다.
경보(alert)는 `flow_id`로 흐름에 붙이고, 정답 라벨은 CIC 라벨 파일과 5-튜플(IP·포트·프로토콜)로 맞춥니다.

**실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 모두 실행.
전체 약 30–60분 (pcap 9 GB 내려받기 수 분, Suricata 분석 15–30분, 학습 수 분). 디스크는 약 20 GB 사용합니다.

## 1. 환경 확인과 설정
`DAY`를 바꾸면 다른 요일을 분석합니다. 요일마다 공격 종류가 다릅니다.

| 요일 | pcap | 공격 |
|---|---|---|
| Tuesday | 약 11 GB | FTP·SSH 무작위 대입(Patator) |
| Wednesday | 약 13 GB | DoS (Hulk, GoldenEye, slowloris, Slowhttptest), Heartbleed |
| Thursday | 약 8 GB | 웹 공격(무작위 대입, XSS, SQL 인젝션), 내부 침투 |
| **Friday** (기본) | 약 9 GB | 봇넷(ARES), 포트 스캔, DDoS(LOIT) |

In [ ]:
import os, sys, re, json, time, random, shutil, subprocess, platform, hashlib, glob, zipfile
IN_COLAB = 'google.colab' in sys.modules
DAY = os.environ.get('IDS_DAY', 'Friday')
WORK = '/content/ids-lab' if IN_COLAB else os.path.abspath(os.environ.get('IDS_LAB_DIR', 'ids-lab'))
HOME_NET = '[192.168.10.0/24]'   # CIC-IDS2017 피해 네트워크. 공격자(205.174.165.x, NAT 후 172.16.0.1)는 외부로 둠
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
RESULTS = os.path.join(WORK, 'results')
os.makedirs(RESULTS, exist_ok=True)

def sh(cmd, check=True, show=True, tail=3000):
    """셸 명령 실행 (Colab·로컬 공통)"""
    if show:
        print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if show and out:
        print(out[-tail:])
    if check and r.returncode != 0:
        raise RuntimeError(f'명령 실패 (코드 {r.returncode}): {cmd}')
    return r

if IN_COLAB:
    sh('pip -q install orjson "huggingface_hub>=0.32" hf_xet', check=False, show=False)
import numpy as np, pandas as pd, sklearn, torch, orjson
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (CPU)',
}
print('분석 요일:', DAY)
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))
sh('nproc; free -g | head -2; df -h / | tail -1', check=False)

## 2. Suricata 설치와 ET Open 룰 받기
OISF 공식 PPA에서 최신 안정판을 설치합니다 (실패하면 Ubuntu 기본 패키지).
룰은 `suricata-update`가 Emerging Threats Open 최신본을 받아 하나의 파일(`suricata.rules`)로 합칩니다.

In [ ]:
if not shutil.which('suricata'):
    sh('apt-get -qq update', check=False, show=False)
    sh('add-apt-repository -y ppa:oisf/suricata-stable', check=False, tail=300)
    sh('DEBIAN_FRONTEND=noninteractive apt-get -qq install -y suricata', tail=300)
if not shutil.which('suricata-update'):
    sh('pip -q install suricata-update', check=False)
SURICATA_VERSION = sh('suricata -V').stdout.strip().split('version')[-1].strip()
VERSIONS['suricata'] = SURICATA_VERSION

RULES_DIR = os.path.join(WORK, 'rules')
os.makedirs(RULES_DIR, exist_ok=True)
RULES = os.environ.get('IDS_RULES') or os.path.join(RULES_DIR, 'suricata.rules')
if not os.path.exists(RULES):
    sh(f'suricata-update --no-test --no-reload -o {RULES_DIR} --data-dir {WORK}/suricata-update-data', tail=800)
rule_lines = [l for l in open(RULES, encoding='utf-8', errors='replace') if l.startswith('alert') or l.startswith('drop')]
RULESET = {
    'file': os.path.basename(RULES), 'enabled_rules': len(rule_lines),
    'sha256': hashlib.sha256(open(RULES, 'rb').read()).hexdigest()[:16],
    'fetched_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime(os.path.getmtime(RULES))),
}
print(json.dumps(RULESET, indent=2))

## 3. 데이터 받기 (CIC-IDS2017)
캐나다 사이버보안연구소(CIC)가 배포한 원본 pcap과 라벨 파일(`GeneratedLabelledFlows.zip`)을
Hugging Face 공개 미러(`bencorn/CICIDS2017`)에서 받습니다. 공식 배포처는 https://www.unb.ca/cic/datasets/ids-2017.html 입니다.

In [ ]:
DATA = os.path.join(WORK, 'data')
os.makedirs(DATA, exist_ok=True)
PCAP = os.environ.get('IDS_PCAP')
LABEL_DIR = os.environ.get('IDS_LABEL_DIR')
if not (PCAP and LABEL_DIR):
    from huggingface_hub import list_repo_files, hf_hub_download
    REPO = 'bencorn/CICIDS2017'
    try:
        files = list_repo_files(REPO, repo_type='dataset')
    except Exception as e:
        raise RuntimeError('Hugging Face 미러에 접근하지 못했습니다. 잠시 후 다시 실행하거나, 공식 배포처에서 '
                           f'{DAY}-WorkingHours.pcap 과 GeneratedLabelledFlows.zip 을 받아 IDS_PCAP / IDS_LABEL_DIR 로 지정하세요. ({e})')
    pcap_name = [f for f in files if os.path.basename(f).lower() == f'{DAY.lower()}-workinghours.pcap']
    zip_name = [f for f in files if os.path.basename(f).lower() == 'generatedlabelledflows.zip']
    if not pcap_name or not zip_name:
        raise RuntimeError(f'미러에서 파일을 찾지 못했습니다. 목록: {files[:30]}')
    t0 = time.time()
    PCAP = hf_hub_download(REPO, pcap_name[0], repo_type='dataset', local_dir=DATA)
    zpath = hf_hub_download(REPO, zip_name[0], repo_type='dataset', local_dir=DATA)
    print(f'내려받기 {time.time()-t0:.0f}초')
    LABEL_DIR = os.path.join(DATA, 'labels')
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            if n.lower().endswith('.csv') and os.path.basename(n).lower().startswith(DAY.lower()):
                z.extract(n, LABEL_DIR)
LABEL_CSVS = sorted(glob.glob(os.path.join(LABEL_DIR, '**', '*.csv'), recursive=True))
PCAP_BYTES = os.path.getsize(PCAP)
print(f'pcap: {os.path.basename(PCAP)}  {PCAP_BYTES/1e9:.2f} GB')
for p in LABEL_CSVS:
    print(f'라벨: {os.path.basename(p):60s} {os.path.getsize(p)/1e6:7.1f} MB')

## 4. Suricata로 pcap 분석
실제 IPS처럼 패킷을 처음부터 끝까지 재생하며 ET Open 룰을 적용합니다. 설정에서 바꾼 것만 적으면:
- `HOME_NET = 192.168.10.0/24` (CIC 피해 네트워크). 기본값(사설망 전체)으로 두면 공격자 172.16.0.1도 내부로 보여 "외부→내부" 룰이 동작하지 않습니다.
- 출력은 `eve.json`에 **경보(alert)와 흐름(flow)** 기록만 남깁니다.
- `-k none`: CIC pcap은 체크섬이 맞지 않는 패킷이 많아 체크섬 검사를 끕니다.

In [ ]:
import yaml
cfg = yaml.safe_load(open('/etc/suricata/suricata.yaml'))
cfg['vars']['address-groups']['HOME_NET'] = HOME_NET
cfg['vars']['address-groups']['EXTERNAL_NET'] = '!$HOME_NET'
for o in cfg['outputs']:
    k = list(o)[0]
    if k == 'eve-log':
        o[k].update({'enabled': True, 'filetype': 'regular', 'filename': 'eve.json',
                     'types': [{'alert': {'tagged-packets': False}}, 'flow']})
    elif isinstance(o[k], dict):
        o[k]['enabled'] = False
cfg.setdefault('flow', {})['memcap'] = '2gb'
cfg.setdefault('stream', {})['memcap'] = '1gb'
cfg['stream']['midstream'] = True
cfg['stream'].setdefault('reassembly', {})['memcap'] = '2gb'
CFG = os.path.join(WORK, 'suricata-lab.yaml')
with open(CFG, 'w') as f:
    f.write('%YAML 1.1\n---\n' + yaml.safe_dump(cfg, sort_keys=False))

OUT = os.path.join(WORK, f'suricata-{DAY}')
EVE = os.path.join(OUT, 'eve.json')
if not os.path.exists(EVE) or os.path.getsize(EVE) == 0:
    shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
    t0 = time.time()
    sh(f'suricata -c {CFG} -S {RULES} -r {PCAP} -l {OUT} -k none --runmode autofp', tail=1500)
    SURI_SEC = round(time.time() - t0, 1)
    json.dump({'sec': SURI_SEC}, open(os.path.join(OUT, 'timing.json'), 'w'))
SURI_SEC = json.load(open(os.path.join(OUT, 'timing.json')))['sec']
SURI_SPEED = {'sec': SURI_SEC, 'mbps': round(PCAP_BYTES * 8 / 1e6 / SURI_SEC, 1)}
print(f'Suricata 분석 {SURI_SEC:.0f}초, 평균 {SURI_SPEED["mbps"]:,} Mbit/s, eve.json {os.path.getsize(EVE)/1e6:.0f} MB')
sh(f'grep -iE "rules successfully loaded|signatures processed|error" {OUT}/suricata.log | head -5', check=False)

## 5. eve.json 읽기 — 흐름 목록과 흐름별 경보
`flow` 기록 하나가 판정 단위입니다. 같은 `flow_id`를 가진 `alert` 기록을 그 흐름에 붙입니다.

In [ ]:
def flags(h):
    try:
        return int(h, 16)
    except (TypeError, ValueError):
        return 0

flows, alerts = [], []
t0 = time.time()
with open(EVE, 'rb') as f:
    for line in f:
        e = orjson.loads(line)
        et = e.get('event_type')
        if et == 'flow':
            fl, tcp = e.get('flow', {}), e.get('tcp', {})
            flows.append((e['flow_id'], e.get('src_ip'), e.get('src_port', 0), e.get('dest_ip'), e.get('dest_port', 0),
                          e.get('proto'), e.get('app_proto', 'none'),
                          fl.get('pkts_toserver', 0), fl.get('pkts_toclient', 0), fl.get('bytes_toserver', 0), fl.get('bytes_toclient', 0),
                          fl.get('start'), fl.get('end'), fl.get('state', ''), fl.get('reason', ''),
                          flags(tcp.get('tcp_flags_ts')), flags(tcp.get('tcp_flags_tc')), tcp.get('state', '')))
        elif et == 'alert':
            a = e['alert']
            alerts.append((e.get('flow_id'), a.get('signature_id'), a.get('signature'), a.get('category', ''), a.get('severity', 3)))
FL = pd.DataFrame(flows, columns=['flow_id', 'src_ip', 'src_port', 'dest_ip', 'dest_port', 'proto', 'app_proto',
                                  'pkts_ts', 'pkts_tc', 'bytes_ts', 'bytes_tc', 'start', 'end', 'state', 'reason',
                                  'flags_ts', 'flags_tc', 'tcp_state'])
AL = pd.DataFrame(alerts, columns=['flow_id', 'sid', 'signature', 'category', 'severity'])
del flows, alerts
print(f'읽기 {time.time()-t0:.0f}초 — 흐름 {len(FL):,}개, 경보 {len(AL):,}건 (서로 다른 룰 {AL.sid.nunique():,}개)')
print(FL.proto.value_counts().to_string())

## 6. 정답 라벨 붙이기
CIC 라벨 파일의 흐름(CICFlowMeter 기준)과 Suricata 흐름은 나누는 방식이 조금 다릅니다.
그래서 **양 끝 IP·포트 + 프로토콜(방향 무시)** 이 같은 CIC 흐름들의 라벨을 모아 Suricata 흐름에 붙입니다.
- 모두 BENIGN → 정상, 공격 라벨만 있음 → 그 공격
- 정상과 공격이 섞임(포트 재사용 등) → **모호**로 빼고 개수를 보고
- CIC 파일에 없는 흐름(ICMP 등) → **매칭 안 됨**으로 빼고 개수를 보고

※ CIC-IDS2017 라벨에는 오류가 있다고 알려져 있습니다 (Engelen 외 2021: 원본 흐름의 20% 이상을 재구성·재라벨).
이 실험은 원본 라벨을 쓰되, 결과를 해석할 때 이 점을 함께 적습니다.

In [ ]:
PROTO_NUM = {'TCP': 6, 'UDP': 17, 'ICMP': 1}

def norm_label(s):
    s = str(s).strip().replace('�', '-').replace('\x96', '-').replace('–', '-')
    return re.sub(r'\s+', ' ', s)

parts = []
for p in LABEL_CSVS:
    d = pd.read_csv(p, encoding='latin-1', low_memory=False, skipinitialspace=True)
    d.columns = [c.strip() for c in d.columns]
    d = d[['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Label']].dropna()
    parts.append(d)
CIC = pd.concat(parts, ignore_index=True)
CIC['Label'] = CIC['Label'].map(norm_label)
for c in ('Source Port', 'Destination Port', 'Protocol'):
    CIC[c] = pd.to_numeric(CIC[c], errors='coerce')
CIC = CIC.dropna()
print(f'CIC 라벨 흐름 {len(CIC):,}개')
print(CIC.Label.value_counts().to_string())

def key_frame(a_ip, a_port, b_ip, b_port, proto):
    a = a_ip.astype(str) + ':' + a_port.astype(int).astype(str)
    b = b_ip.astype(str) + ':' + b_port.astype(int).astype(str)
    lo, hi = np.where(a <= b, a, b), np.where(a <= b, b, a)
    return pd.Series(lo, index=a.index) + '|' + pd.Series(hi, index=a.index) + '|' + proto.astype(int).astype(str)

CIC['key'] = key_frame(CIC['Source IP'], CIC['Source Port'], CIC['Destination IP'], CIC['Destination Port'], CIC['Protocol'])
agg = CIC.groupby('key')['Label'].agg(lambda s: sorted(set(s)))
def resolve(labels):
    attacks = [l for l in labels if l != 'BENIGN']
    if not attacks:
        return 'BENIGN'
    if len(attacks) == len(labels) and len(set(attacks)) == 1:
        return attacks[0]
    return 'AMBIGUOUS'
KEY_LABEL = agg.map(resolve)
AMBIG = agg[KEY_LABEL == 'AMBIGUOUS'].map(lambda l: ' + '.join(l))

FL['proto_num'] = FL.proto.map(PROTO_NUM).fillna(0).astype(int)
FL['key'] = key_frame(FL.src_ip, FL.src_port.fillna(0), FL.dest_ip, FL.dest_port.fillna(0), FL.proto_num)
FL['attack'] = FL.key.map(KEY_LABEL).fillna('UNMATCHED')
LABEL_STATS = FL.attack.value_counts().to_dict()
print('\nSuricata 흐름에 붙은 라벨:')
print(FL.attack.value_counts().to_string())
df = FL[~FL.attack.isin(['UNMATCHED', 'AMBIGUOUS'])].reset_index(drop=True)
df['label'] = (df.attack != 'BENIGN').astype(int)
amb = FL[FL.attack == 'AMBIGUOUS'].key.map(AMBIG).value_counts()
print('\n모호 흐름의 라벨 조합:'); print(amb.head(8).to_string())
amb.to_csv(os.path.join(RESULTS, 'ambiguous_label_mix.csv'))
print(f'\n비교에 쓰는 흐름 {len(df):,}개 (정상 {int((df.label==0).sum()):,} / 공격 {int(df.label.sum()):,}),'
      f' 매칭 안 됨 {LABEL_STATS.get("UNMATCHED",0):,}, 모호 {LABEL_STATS.get("AMBIGUOUS",0):,} 제외')

## 7. 룰 판정 — 흐름에 경보가 붙었는가
`suricata-update`가 만든 룰 파일에는 ET Open 룰과 함께 **Suricata 자체 프로토콜 이상 룰**(sid 2200000–2299999:
체크섬 오류, TCP 스트림 이상, HTTP 파싱 이상 등)이 들어 있습니다. 이것은 공격 탐지가 아니라 패킷 품질 경보라 따로 셉니다.
ET 룰에도 "정보성·정책" 경보가 많아 심각도로 나눠 봅니다.
- **ET 전체**: ET 룰 경보가 하나라도 붙으면 탐지
- **ET 심각도 1–2**: 높음·중간만 (운영에서 주로 보는 기준)
- **ET 심각도 1**: 높음만
- (참고) **Suricata 전체**: 프로토콜 이상 룰까지 포함

In [ ]:
AL['engine_event'] = AL.sid.between(2200000, 2299999)
print(f"경보 {len(AL):,}건 중 Suricata 프로토콜 이상 룰 {int(AL.engine_event.sum()):,}건, ET 룰 {int((~AL.engine_event).sum()):,}건")
print(AL[AL.engine_event].signature.value_counts().head(5).to_string())
fa = AL.dropna(subset=['flow_id']).groupby('flow_id').agg(min_sev_all=('severity', 'min'))
et = AL[~AL.engine_event].dropna(subset=['flow_id']).groupby('flow_id').agg(
    min_sev=('severity', 'min'), n_alerts=('sid', 'size'), sids=('sid', lambda s: ' '.join(map(str, sorted(set(s))))))
fa = fa.join(et, how='left')
df = df.merge(fa, left_on='flow_id', right_index=True, how='left')
df['n_alerts'] = df.n_alerts.fillna(0).astype(int)
df['sids'] = df.sids.fillna('')
RULE_VARIANTS = {'ET 전체': 3, 'ET 심각도 1–2': 2, 'ET 심각도 1': 1}
for name, sv in RULE_VARIANTS.items():
    df[name] = df.min_sev.le(sv)
df['Suricata 전체(이상 룰 포함)'] = df.min_sev_all.le(3)
RULE_VARIANTS['Suricata 전체(이상 룰 포함)'] = 3
print(df.groupby('attack')[list(RULE_VARIANTS)].mean().round(4).to_string())

## 8. 학습 / 검증 / 시험 분할과 AI 특성
흐름을 공격 종류 비율을 유지한 채 **60 / 20 / 20**으로 나눕니다. 모든 비교는 시험 집합에서만 합니다.

AI 특성은 Suricata 흐름 기록의 숫자만 씁니다: 프로토콜, 목적지 포트, 응용 프로토콜, 방향별 패킷·바이트 수,
지속 시간, 패킷당 바이트, TCP 플래그, 연결 상태. **IP 주소와 경보 여부는 넣지 않습니다** (정답을 외우거나 룰 결과를 베끼지 않도록).

In [ ]:
from sklearn.model_selection import train_test_split
idx_tr, idx_tmp = train_test_split(df.index, test_size=0.4, stratify=df.attack, random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.5, stratify=df.attack[idx_tmp], random_state=SEED)
df['split'] = 'train'; df.loc[idx_va, 'split'] = 'val'; df.loc[idx_te, 'split'] = 'test'
tr, va, te = (df.split == 'train').values, (df.split == 'val').values, (df.split == 'test').values
print(df.groupby(['split', 'label']).size().unstack().rename(columns={0: 'normal', 1: 'attack'}))

start = pd.to_datetime(df.start, utc=True, errors='coerce', format='ISO8601')
end = pd.to_datetime(df.end, utc=True, errors='coerce', format='ISO8601')
X = pd.DataFrame({
    'proto': df.proto_num, 'dest_port': df.dest_port.fillna(0), 'src_port_eph': (df.src_port.fillna(0) >= 1024).astype(int),
    'pkts_ts': df.pkts_ts, 'pkts_tc': df.pkts_tc, 'bytes_ts': df.bytes_ts, 'bytes_tc': df.bytes_tc,
    'duration': (end - start).dt.total_seconds().fillna(0).clip(lower=0),
    'bpp_ts': df.bytes_ts / df.pkts_ts.clip(lower=1), 'bpp_tc': df.bytes_tc / df.pkts_tc.clip(lower=1),
    'pkt_ratio': df.pkts_tc / df.pkts_ts.clip(lower=1), 'flags_ts': df.flags_ts, 'flags_tc': df.flags_tc,
})
for col in ('app_proto', 'state', 'reason', 'tcp_state'):
    X[col] = df[col].astype('category').cat.codes
X['pps'] = (df.pkts_ts + df.pkts_tc) / X.duration.clip(lower=1e-3)
FEATURES = list(X.columns)
print('특성', len(FEATURES), '개:', FEATURES)

## 9. AI 모델 A — 그래디언트 부스팅 (CPU)
표 형태 데이터에서 가장 많이 쓰는 방식입니다. 공격 종류는 구분하지 않고 "정상 / 공격"만 학습합니다.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
TRAIN_TIME = {}
def make_a():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63,
                                          early_stopping=True, validation_fraction=0.1, random_state=SEED)
t0 = time.time()
A = make_a().fit(X[tr], df.label[tr])
TRAIN_TIME['A_gbdt'] = round(time.time() - t0, 1)
t0 = time.time(); df['a_score'] = A.predict_proba(X)[:, 1]
A_SPEED = round(len(df) / (time.time() - t0), 1)
print(f'학습 {TRAIN_TIME["A_gbdt"]}초, 판정 {A_SPEED:,} 흐름/초, 검증 AUC {roc_auc_score(df.label[va], df.a_score[va]):.4f}')

### 모델 A가 무엇을 보고 판단했나 (순열 중요도)
특성 하나를 무작위로 섞었을 때 성능(AUC)이 얼마나 떨어지는지로 중요도를 잽니다. 시험 집합 2만 개 표본 사용.

In [ ]:
from sklearn.inspection import permutation_importance
samp = np.random.RandomState(SEED).choice(np.where(te)[0], size=min(20000, int(te.sum())), replace=False)
pi = permutation_importance(A, X.iloc[samp], df.label.iloc[samp], scoring='roc_auc', n_repeats=3, random_state=SEED)
IMPORTANCE = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print(IMPORTANCE.head(10).round(4).to_string())
IMPORTANCE.to_csv(os.path.join(RESULTS, 'feature_importance_A.csv'))

## 10. AI 모델 B — 다층 신경망 MLP (GPU)
같은 특성을 로그 변환·표준화해 3층 신경망으로 학습합니다.

In [ ]:
import torch.nn as nn
Xn = np.log1p(X.clip(lower=0).astype('float64')).values
mu, sd = Xn[tr].mean(0), Xn[tr].std(0) + 1e-6
Xn = ((Xn - mu) / sd).astype('float32')
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 128), nn.ReLU(),
                                 nn.Dropout(0.2), nn.Linear(128, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)
model = MLP(Xn.shape[1]).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()
Xt, yt_ = torch.tensor(Xn[tr]), torch.tensor(df.label[tr].values, dtype=torch.float32)
def predict(Xa, bs=65536):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xa), bs):
            out.append(torch.sigmoid(model(torch.tensor(Xa[i:i+bs]).to(DEVICE))).cpu().numpy())
    return np.concatenate(out)
EPOCHS = int(os.environ.get('IDS_EPOCHS', 8))
t0 = time.time(); best_auc, best_state = 0, None
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(len(Xt)); tot = 0
    for i in range(0, len(perm), 2048):
        b = perm[i:i+2048]
        xb, yb = Xt[b].to(DEVICE), yt_[b].to(DEVICE)
        opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step(); tot += loss.item() * len(b)
    auc = roc_auc_score(df.label[va], predict(Xn[va]))
    print(f'epoch {ep+1}/{EPOCHS}  loss {tot/len(perm):.4f}  val AUC {auc:.4f}')
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
TRAIN_TIME['B_mlp'] = round(time.time() - t0, 1)
t0 = time.time(); df['b_score'] = predict(Xn)
B_SPEED = round(len(df) / (time.time() - t0), 1)
print(f'학습 {TRAIN_TIME["B_mlp"]}초 ({DEVICE}), 판정 {B_SPEED:,} 흐름/초')

## 11. 비교 — 같은 오탐률에서 탐지율
AI 판정 기준값은 **검증 집합**에서 오탐률이 정해진 값이 되도록 고른 뒤 시험 집합에 적용합니다.
"룰과 같은 오탐률"은 ET 심각도 1–2 경보의 검증 집합 오탐률입니다.

In [ ]:
def thr_at_fpr(scores_neg, target):
    s = np.sort(scores_neg)[::-1]
    k = int(np.floor(target * len(s)))
    return s[k] + 1e-9 if k < len(s) else s[-1]

def metrics(name, yt, pred, extra=None):
    yt, pred = np.asarray(yt).astype(int), np.asarray(pred).astype(bool)
    tp = int((pred & (yt == 1)).sum()); fn = int((~pred & (yt == 1)).sum())
    fp = int((pred & (yt == 0)).sum()); tn = int((~pred & (yt == 0)).sum())
    tpr = tp / max(tp + fn, 1); fpr = fp / max(fp + tn, 1); prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * tpr / max(prec + tpr, 1e-9)
    d = dict(model=name, detection_rate=round(tpr, 4), false_positive_rate=round(fpr, 5),
             precision=round(prec, 4), f1=round(f1, 4), TP=tp, FN=fn, FP=fp, TN=tn)
    if extra:
        d.update(extra)
    return d

yt = df.label[te].values
rows = [metrics(n, yt, df[n][te], {'kind': 'rule'}) for n in RULE_VARIANTS]
rule_val_fpr = float(df['ET 심각도 1–2'][va & (df.label == 0)].mean())
targets = {'FPR 1%': 0.01, 'FPR 0.1%': 0.001, f'룰과 같은 FPR ({rule_val_fpr:.3%})': max(rule_val_fpr, 1e-5)}
THRESH = {}
for col, nm, spd in (('a_score', 'A GBDT', A_SPEED), ('b_score', 'B MLP', B_SPEED)):
    for tname, target in targets.items():
        th = thr_at_fpr(df[col][va & (df.label == 0)].values, target)
        THRESH[f'{nm} @ {tname}'] = float(th)
        rows.append(metrics(f'{nm} @ {tname}', yt, df[col][te] >= th, {'kind': 'ai', 'speed_flows_per_s': spd}))
thA = THRESH['A GBDT @ FPR 0.1%']
df['hybrid'] = df['ET 심각도 1–2'] | (df.a_score >= thA)
rows.append(metrics('Hybrid: ET 심각도1–2 OR A@FPR0.1%', yt, df.hybrid[te], {'kind': 'hybrid'}))
cmp = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(cmp[['model', 'detection_rate', 'false_positive_rate', 'precision', 'f1', 'TP', 'FN', 'FP', 'TN']].to_string(index=False))
cmp.to_csv(os.path.join(RESULTS, 'comparison.csv'), index=False)

## 12. 공격 종류별 — 흐름 단위 탐지율과 "경보가 한 번이라도 났는가"
IPS 룰은 스캔·무작위 대입처럼 같은 행동이 반복되면 **일정 횟수마다 경보를 한 번** 내도록(threshold) 만든 것이 많습니다.
그래서 흐름 단위 탐지율은 낮아도 **공격 자체는 알아챈** 경우가 있습니다. 두 관점을 함께 봅니다.

In [ ]:
tt = df[te]
per = tt.groupby('attack').agg(flows=('label', 'size'),
                               rule_all=('ET 전체', 'mean'), rule_sev12=('ET 심각도 1–2', 'mean'),
                               ai_A=('a_score', lambda s: float((s >= thA).mean())),
                               ai_B=('b_score', lambda s: float((s >= THRESH['B MLP @ FPR 0.1%']).mean())))
ev = df.groupby('attack').agg(all_flows=('label', 'size'), alerted_flows=('n_alerts', lambda s: int((s > 0).sum())))
per = per.join(ev)
sig = AL[~AL.engine_event].merge(df[['flow_id', 'attack']], on='flow_id')
top = sig.groupby(['attack', 'signature']).size().reset_index(name='n').sort_values(['attack', 'n'], ascending=[True, False])
per['top_signatures'] = pd.Series({a: ' | '.join(f'{s} ({n})' for s, n in zip(g.signature[:3], g.n[:3])) for a, g in top.groupby('attack')})
per = per.sort_values('flows', ascending=False)
print(per.drop(columns='top_signatures').round(4).to_string())
print()
for a, s in per.top_signatures.dropna().items():
    print(f'[{a}] {s}')
per.to_csv(os.path.join(RESULTS, 'by_attack.csv'))

## 13. 정상 흐름에 가장 많이 난 경보 — 룰 튜닝 대상
실제 운영에서 IPS 오탐은 이 표를 보고 **룰을 끄거나 예외를 거는 방식**으로 줄입니다.

In [ ]:
fp_sig = sig[sig.attack == 'BENIGN'].groupby(['sid', 'signature', 'category']).size().reset_index(name='benign_flows')
sev = AL.drop_duplicates('sid').set_index('sid').severity
fp_sig['severity'] = fp_sig.sid.map(sev)
ev_sig = AL[AL.engine_event].merge(df[['flow_id', 'attack']], on='flow_id').groupby(['signature', 'attack']).size().unstack(fill_value=0)
ev_sig.to_csv(os.path.join(RESULTS, 'engine_event_alerts.csv'))
print('Suricata 프로토콜 이상 룰 (흐름 수):'); print(ev_sig.loc[ev_sig.sum(axis=1).sort_values(ascending=False).index].head(6).to_string()); print()
fp_sig = fp_sig.sort_values('benign_flows', ascending=False)
print(fp_sig.head(15).to_string(index=False))
fp_sig.to_csv(os.path.join(RESULTS, 'benign_alert_signatures.csv'), index=False)

## 14. 처음 보는 공격도 잡는가 — 공격 종류 하나씩 빼고 학습
지도 학습 AI는 학습에 없던 공격을 잘 못 잡을 수 있습니다. 룰은 학습이 필요 없으므로 이 점이 중요한 차이입니다.
공격 종류 하나를 **학습·검증 집합에서 통째로 빼고** 모델 A를 다시 학습한 뒤, 시험 집합의 그 공격을 얼마나 잡는지 봅니다 (기준값: 오탐률 0.1%).

In [ ]:
loo = []
for atk, n in tt.attack.value_counts().items():
    if atk == 'BENIGN' or n < 20:
        continue
    keep_tr = tr & (df.attack != atk).values
    m = make_a().fit(X[keep_tr], df.label[keep_tr])
    th = thr_at_fpr(m.predict_proba(X[va & (df.label == 0).values])[:, 1], 0.001)
    s = m.predict_proba(X[te & (df.attack == atk).values])[:, 1]
    loo.append({'attack': atk, 'test_flows': int(n), 'A_seen': round(float(per.loc[atk, 'ai_A']), 4),
                'A_unseen': round(float((s >= th).mean()), 4), 'rule_sev12': round(float(per.loc[atk, 'rule_sev12']), 4)})
    print(loo[-1])
LOO = pd.DataFrame(loo)
LOO.to_csv(os.path.join(RESULTS, 'unseen_attack.csv'), index=False)

## 15. ROC 곡선

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for col, nm in (('a_score', 'A GBDT'), ('b_score', 'B MLP')):
    fpr, tpr, _ = roc_curve(yt, df[col][te])
    ax.plot(fpr, tpr, label=f'{nm} (AUC {roc_auc_score(yt, df[col][te]):.3f})')
EN = {'ET 전체': 'ET Open: all alerts', 'ET 심각도 1–2': 'ET Open: severity 1-2', 'ET 심각도 1': 'ET Open: severity 1',
      'Suricata 전체(이상 룰 포함)': 'Suricata: incl. protocol-anomaly rules'}
for (n, mk) in zip(RULE_VARIANTS, ('o', 's', 'D', 'X')):
    r = cmp[cmp.model == n].iloc[0]
    ax.scatter(r.false_positive_rate, r.detection_rate, marker=mk, s=80, zorder=5, label=EN[n])
r = cmp[cmp.kind == 'hybrid'].iloc[0]
ax.scatter(r.false_positive_rate, r.detection_rate, marker='*', s=220, c='k', zorder=6, label='Hybrid (ET sev1-2 OR A)')
ax.set_xscale('symlog', linthresh=0.01); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel('False positive rate (log scale above 1%)'); ax.set_ylabel('Detection rate (per flow)')
ax.set_title(f'CIC-IDS2017 {DAY}: Suricata + ET Open vs AI'); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'roc.png'), dpi=130); plt.show()

## 16. 사례 저장과 해석할 때 주의할 점
- **흐름 단위 탐지율은 룰에 불리합니다.** 스캔·DDoS처럼 흐름 수천 개로 이뤄진 공격은 경보가 일부 흐름에만 붙어도 운영자는 공격을 알아챕니다 (12번 표의 `alerted_flows`).
- **AI는 이 네트워크·이 날의 패턴을 배웠습니다.** 같은 날 안에서 나눴기 때문에 학습과 시험에 비슷한 공격 흐름이 함께 들어 있고, 다른 네트워크에서는 이 점수가 나오지 않습니다. 14번(처음 보는 공격)이 더 현실에 가까운 시험입니다.
- **라벨 오류**: CIC-IDS2017 원본 라벨에는 알려진 오류가 있습니다. 공격 시간대의 실패한 연결이 공격으로, 일부 공격 흐름이 정상으로 붙은 경우가 있어 "AI만 잡음 / 룰 오탐" 일부는 라벨 문제일 수 있습니다.
- **ET Open 룰은 계속 바뀝니다.** 실행한 날의 룰 버전(sha256)을 결과에 기록합니다.

In [ ]:
def case(mask, n=8):
    cols = ['src_ip', 'src_port', 'dest_ip', 'dest_port', 'proto', 'app_proto', 'attack', 'pkts_ts', 'pkts_tc', 'bytes_ts', 'bytes_tc', 'sids', 'a_score']
    return tt[mask][cols].sort_values('a_score', ascending=False)
rule = tt['ET 심각도 1–2']; ai = tt.a_score >= thA; y = tt.label == 1
CASES = {'ai_only_detected': case(y & ai & ~rule), 'rule_only_detected': case(y & rule & ~ai),
         'both_missed': case(y & ~ai & ~rule), 'rule_false_positive': case(~y & rule), 'ai_false_positive': case(~y & ai)}
CASE_COUNTS = {k: len(v) for k, v in CASES.items()}
for k, v in CASES.items():
    v.to_csv(os.path.join(RESULTS, f'cases_{k}.csv'), index=False)
    print(f'## {k}: {len(v):,}건  공격 종류: {v.attack.value_counts().head(4).to_dict()}')

## 17. 결과 저장
`results/` 폴더를 압축해 내려받습니다.

In [ ]:
summary = {
    'dataset': f'CIC-IDS2017 {DAY}', 'pcap_gb': round(PCAP_BYTES / 1e9, 2), 'label_stats': LABEL_STATS,
    'split': df.groupby(['split', 'label']).size().unstack().to_dict(),
    'versions': VERSIONS, 'ruleset': RULESET, 'suricata_speed': SURI_SPEED,
    'train_time_sec': TRAIN_TIME, 'speed_flows_per_s': {'A': A_SPEED, 'B': B_SPEED},
    'features': FEATURES, 'thresholds': THRESH, 'case_counts': CASE_COUNTS, 'mlp_best_val_auc': round(float(best_auc), 4),
    'importance_A': IMPORTANCE.round(4).to_dict(), 'comparison': cmp.to_dict(orient='records'), 'unseen_attack': LOO.to_dict(orient='records'),
}
json.dump(summary, open(os.path.join(RESULTS, 'summary.json'), 'w'), ensure_ascii=False, indent=2, default=str)
zip_path = shutil.make_archive(os.path.join(WORK, f'ids_lab_results_{DAY}'), 'zip', RESULTS)
print('저장:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB')
print(sorted(os.listdir(RESULTS)))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)